# MLP Backprop by Hand (No Autograd) — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

Every deep-learning framework is, underneath, the chain rule applied one layer at a time. Writing the backward pass of a small network by hand (the CS231n "two-layer net" exercise; Karpathy's *"Yes you should understand backprop"*) is a classic interview question because it exposes whether you really know which shapes flow backwards, what the forward pass must cache, and why softmax + cross-entropy has such a simple gradient. It is also the mental model you need to debug exploding/vanishing gradients, write a custom `torch.autograd.Function`, or reason about activation memory and gradient checkpointing.

Build an MLP classifier from three hand-differentiated pieces — `Linear`, `ReLU` and a fused softmax-cross-entropy loss — and train it with plain SGD on a two-spirals toy dataset. **No autograd anywhere.**

---

### Requirements

1. **`Linear(in_features, out_features)`** — a plain Python class (not an `nn.Module`).
   - Parameters are plain tensors `self.W` of shape `(out_features, in_features)` (the same layout as `nn.Linear.weight`) and `self.b` of shape `(out_features,)`.
   - **He/Kaiming-uniform init:** $W \sim \mathcal{U}(-a, a)$ with $a = \sqrt{6/d_\text{in}}$ (where $d_\text{in}$ = `in_features` = fan-in), so that $\mathrm{Var}(W) = 2/d_\text{in}$; $b = 0$. (The grader checks the zero mean and the variance $2/d_\text{in}$, so He-normal also passes.)
   - `forward(x)`: `x` is `(N, in_features)`; return $y = xW^\top + b$, shape `(N, out_features)`, and cache whatever `backward` needs.
   - `backward(grad_out)`: `grad_out` is $\partial L/\partial y$, shape `(N, out_features)`. Store `self.grad_W` `(out, in)` and `self.grad_b` `(out,)` — **overwrite, don't accumulate** — and return $\partial L/\partial x$, shape `(N, in_features)`.
   - Read `self.W` / `self.b` at call time: the grader assigns its own (float64) tensors to them.
2. **`ReLU()`** — `forward(x)` returns $\max(x, 0)$; `backward(grad_out)` returns $\partial L/\partial x$. At exactly $x = 0$ the gradient is **0** (PyTorch's convention: the local gradient is $\mathbb{1}[x > 0]$).
3. **`softmax_cross_entropy(logits, targets) -> (loss, grad_logits)`**
   - `logits`: `(N, C)`; `targets`: `(N,)` int64 class indices.
   - `loss`: scalar tensor, the batch **mean** of $-\log \mathrm{softmax}(z_n)_{y_n}$ — identical to `F.cross_entropy(logits, targets)`.
   - `grad_logits`: $\partial\,\text{loss} / \partial z = \big(\mathrm{softmax}(z) - \mathrm{onehot}(y)\big)/N$, shape `(N, C)`.
   - Numerically stable: finite and correct for logits of magnitude $10^3$ (exact float64 agreement with `F.cross_entropy`).
4. **`MLP(sizes)`** — e.g. `MLP([2, 32, 32, 2])` is `Linear(2,32) → ReLU → Linear(32,32) → ReLU → Linear(32,2)`; **no ReLU on the logits**.
   - Keep the layers, in forward order, in a list `self.layers`.
   - `forward(x)` → logits `(N, sizes[-1])`.
   - `backward(grad_logits)` → runs every layer's `backward` in reverse order and returns $\partial L/\partial x$.
   - `step(lr)` → plain SGD on every `Linear`: `W -= lr * grad_W`, `b -= lr * grad_b`.
5. Everything must work in float32 and float64 under `torch.no_grad()` on tensors with `requires_grad=False` — that is how the grader runs it.

---

### Constraints

- ✅ Plain tensor ops: `@`, `.T`, `.sum`, `.max`, `.exp`, `.log`, `.clamp`, advanced indexing, `torch.arange`, `torch.logsumexp`.
- ❌ No autograd inside your code: no `.backward()`, `torch.autograd.*`, `requires_grad_()`.
- ❌ No `nn.Linear`, `F.cross_entropy`, `F.softmax` / `F.log_softmax`, `torch.optim` in the implementation (the tests use them as the oracle).

---

<details>
  <summary>💡 Hint</summary>

  - Draw the shapes. A gradient always has exactly the shape of the tensor it is the gradient of: `grad_W` is `(out, in)`. Which product of `grad_out (N, out)` and `x (N, in)` has that shape?
  - The bias is added to every row of the batch. What must happen to the $N$ per-row contributions in the backward pass?
  - For the loss, write $\ell = \log\sum_j e^{z_j} - z_y$ and differentiate w.r.t. $z_k$ directly — you never need the full softmax Jacobian.
  - $\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}$ for any $m$. Which $m$ makes every exponent $\le 0$?
  - The $1/N$ of the mean belongs to the loss gradient only — the layers just apply the chain rule to whatever arrives.

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F   # used by the tests as an oracle, never inside your implementation

In [ ]:
def make_spirals(n_per_class=100, noise=0.15, turns=1.25, seed=0):
    """Two interleaved spirals in [-1, 1]^2 — a classic not-linearly-separable toy set.

    Returns:
        X: (2 * n_per_class, 2) float32 points
        y: (2 * n_per_class,) int64 labels in {0, 1}
    """
    g = torch.Generator().manual_seed(seed)
    X, y = [], []
    for c in range(2):
        r = torch.linspace(0.1, 1.0, n_per_class)
        t = (torch.linspace(0, turns * 2 * math.pi, n_per_class) + c * math.pi
             + noise * torch.randn(n_per_class, generator=g))
        X.append(torch.stack([r * torch.cos(t), r * torch.sin(t)], dim=1))
        y.append(torch.full((n_per_class,), c, dtype=torch.long))
    return torch.cat(X), torch.cat(y)


X, y = make_spirals()
print(f"X {tuple(X.shape)}, y {tuple(y.shape)}, class counts {y.bincount().tolist()}")

In [ ]:
class Linear:
    """Fully connected layer y = x @ W.T + b with a hand-written backward pass."""

    def __init__(self, in_features: int, out_features: int):
        # He/Kaiming-uniform. For y = W x with ReLU inputs, Var(y) = fan_in * Var(W) * E[x^2] and
        # ReLU halves E[x^2], so Var(W) = 2 / fan_in keeps the activation scale constant with depth.
        # U(-a, a) has variance a^2 / 3  =>  a = sqrt(6 / fan_in).
        bound = math.sqrt(6.0 / in_features)
        self.W = (torch.rand(out_features, in_features) * 2 - 1) * bound   # (out, in) like nn.Linear
        self.b = torch.zeros(out_features)
        self.grad_W = None
        self.grad_b = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.x = x                                    # (N, in): needed for grad_W
        return x @ self.W.T + self.b                  # (N, out)

    def backward(self, grad_out: torch.Tensor) -> torch.Tensor:
        self.grad_W = grad_out.T @ self.x             # (out, N) @ (N, in) -> (out, in)
        self.grad_b = grad_out.sum(dim=0)             # b was broadcast over the batch -> sum
        return grad_out @ self.W                      # (N, out) @ (out, in) -> (N, in)

    def __call__(self, x):
        return self.forward(x)


class ReLU:
    """max(x, 0) with a hand-written backward pass (gradient 0 at x == 0, like PyTorch)."""

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.mask = x > 0                             # strict: the gradient at exactly 0 is 0
        return x.clamp(min=0)

    def backward(self, grad_out: torch.Tensor) -> torch.Tensor:
        return grad_out * self.mask

    def __call__(self, x):
        return self.forward(x)

In [ ]:
def softmax_cross_entropy(logits: torch.Tensor, targets: torch.Tensor):
    N = logits.shape[0]
    rows = torch.arange(N)
    shifted = logits - logits.max(dim=1, keepdim=True).values         # every entry <= 0, max = 0
    log_probs = shifted - shifted.exp().sum(dim=1, keepdim=True).log() # log-softmax; the sum is >= 1
    loss = -log_probs[rows, targets].mean()
    grad = log_probs.exp()                                             # softmax, (N, C)
    grad[rows, targets] -= 1.0                                         # softmax - onehot
    return loss, grad / N                                              # the 1/N of the mean

In [ ]:
class MLP:
    """Linear -> ReLU -> ... -> Linear (no ReLU on the logits), trained with plain SGD."""

    def __init__(self, sizes):
        self.layers = []
        for i, (fan_in, fan_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(fan_in, fan_out))
            if i < len(sizes) - 2:                    # hidden layers only
                self.layers.append(ReLU())

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad: torch.Tensor) -> torch.Tensor:
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def step(self, lr: float) -> None:
        for layer in self.layers:
            if isinstance(layer, Linear):
                layer.W -= lr * layer.grad_W
                layer.b -= lr * layer.grad_b

    def __call__(self, x):
        return self.forward(x)

In [ ]:
# Test 1 — Linear: output and all three gradients vs torch.autograd (float64)
torch.manual_seed(0)
lin = Linear(5, 4)
assert tuple(lin.W.shape) == (4, 5) and tuple(lin.b.shape) == (4,), "W must be (out, in), b (out,)"
lin.W = torch.randn(4, 5, dtype=torch.float64)
lin.b = torch.randn(4, dtype=torch.float64)
x = torch.randn(7, 5, dtype=torch.float64)
g = torch.randn(7, 4, dtype=torch.float64)            # an arbitrary upstream gradient dL/dy

with torch.no_grad():                                  # your code must not need autograd
    out = lin.forward(x)
    gx = lin.backward(g)

W_ref, b_ref, x_ref = (t.clone().requires_grad_() for t in (lin.W, lin.b, x))
F.linear(x_ref, W_ref, b_ref).backward(g)

assert torch.allclose(out, F.linear(x, lin.W, lin.b)), "forward is wrong"
assert torch.allclose(lin.grad_W, W_ref.grad), "grad_W is wrong"
assert torch.allclose(lin.grad_b, b_ref.grad), "grad_b is wrong"
assert torch.allclose(gx, x_ref.grad), "dL/dx is wrong"
print("Linear: forward and gradients match autograd")

In [ ]:
# Test 2 — ReLU, including the gradient at exactly 0
relu = ReLU()
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 3.0], dtype=torch.float64)
with torch.no_grad():
    out = relu.forward(x)
    gx = relu.backward(torch.ones_like(x))
assert torch.allclose(out, torch.tensor([0.0, 0.0, 0.0, 0.5, 3.0], dtype=torch.float64))
assert torch.allclose(gx, torch.tensor([0.0, 0.0, 0.0, 1.0, 1.0], dtype=torch.float64)), \
    f"got {gx.tolist()} — the gradient at exactly 0 must be 0 (PyTorch's convention)"
print("ReLU OK")

In [ ]:
# Test 3 — softmax cross-entropy vs F.cross_entropy, on normal and extreme logits
torch.manual_seed(0)
cases = {
    "random": (3 * torch.randn(6, 5, dtype=torch.float64), torch.randint(0, 5, (6,))),
    "extreme": (torch.tensor([[1000.0, -1000.0, 0.0],
                              [-1000.0, 1000.0, 1000.0],
                              [999.0, 1000.0, -1000.0]], dtype=torch.float64),
                torch.tensor([1, 0, 0])),
}
for name, (z, t) in cases.items():
    with torch.no_grad():
        loss, grad = softmax_cross_entropy(z, t)
    z_ref = z.clone().requires_grad_()
    ref = F.cross_entropy(z_ref, t)
    ref.backward()
    assert torch.isfinite(torch.as_tensor(loss)), f"{name}: loss is {loss}"
    assert torch.allclose(torch.as_tensor(loss, dtype=torch.float64), ref.detach()), \
        f"{name}: loss {float(loss):.6f} vs F.cross_entropy {ref.item():.6f}"
    assert torch.allclose(grad, z_ref.grad), f"{name}: grad_logits is wrong"
    print(f"{name:8s} logits: loss {float(loss):10.4f} — matches F.cross_entropy, gradient OK")

In [ ]:
# Test 4 — whole MLP: every gradient vs autograd on an equivalent graph (float64)
torch.manual_seed(0)
mlp = MLP([6, 9, 7, 4])
linears = [layer for layer in mlp.layers if isinstance(layer, Linear)]
assert len(linears) == 3
for layer in linears:                                  # float64 for a tight comparison
    layer.W, layer.b = layer.W.double(), (0.1 * torch.randn_like(layer.b)).double()
x = torch.randn(10, 6, dtype=torch.float64)
t = torch.randint(0, 4, (10,))

with torch.no_grad():
    logits = mlp.forward(x)
    loss, grad_logits = softmax_cross_entropy(logits, t)
    gx = mlp.backward(grad_logits)

params = [p.clone().requires_grad_() for layer in linears for p in (layer.W, layer.b)]
x_ref = x.clone().requires_grad_()
h = x_ref
for k in range(len(linears)):
    h = F.linear(h, params[2 * k], params[2 * k + 1])
    if k < len(linears) - 1:
        h = F.relu(h)
loss_ref = F.cross_entropy(h, t)
loss_ref.backward()

assert torch.allclose(logits, h.detach()), "MLP forward is wrong (is there a ReLU on the logits?)"
for k, layer in enumerate(linears):
    assert torch.allclose(layer.grad_W, params[2 * k].grad), f"grad_W of layer {k} is wrong"
    assert torch.allclose(layer.grad_b, params[2 * k + 1].grad), f"grad_b of layer {k} is wrong"
assert torch.allclose(gx, x_ref.grad), "dL/dx returned by MLP.backward is wrong"
print("MLP: all gradients match autograd")

In [ ]:
# Test 5 — train on the two spirals with full-batch SGD
torch.manual_seed(0)
model = MLP([2, 32, 32, 2])
losses = []
with torch.no_grad():
    for step in range(300):
        loss, grad = softmax_cross_entropy(model.forward(X), y)
        model.backward(grad)
        model.step(lr=0.5)
        losses.append(float(loss))
    acc = (model.forward(X).argmax(dim=1) == y).float().mean().item()
print(f"loss {losses[0]:.3f} -> {losses[-1]:.4f}, train accuracy {acc:.1%}")
assert losses[-1] < 0.3 * losses[0] and acc > 0.95, "the MLP is not learning the spirals"

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("mlp-backprop", Linear=Linear, ReLU=ReLU, softmax_cross_entropy=softmax_cross_entropy, MLP=MLP)

### Gradient check with finite differences

Before trusting a hand-written backward pass, compare it with a numerical estimate. The central difference
$\frac{f(\theta + \epsilon e_i) - f(\theta - \epsilon e_i)}{2\epsilon}$ has truncation error $O(\epsilon^2)$ and round-off error
$O(\varepsilon_{\text{mach}}/\epsilon)$, so in float64 with $\epsilon \approx 10^{-6}$ a correct gradient gives a relative error around
$10^{-8}$ or smaller; a bug typically gives $10^{-2}$–$1$. (In float32 the round-off term alone is ~$10^{-2}$ — always grad-check in float64.
Avoid points where a ReLU input sits within $\epsilon$ of 0: the function has a kink there.) `torch.autograd.gradcheck` does exactly this.

In [ ]:
def numerical_grad(f, t, eps=1e-6):
    """Central-difference estimate of d f() / d t, where f() is a scalar function that reads t.

    Perturbs t in place, one entry at a time (restoring it afterwards). Use float64.
    """
    grad = torch.zeros_like(t)
    flat, gflat = t.view(-1), grad.view(-1)
    for i in range(flat.numel()):
        orig = flat[i].item()
        flat[i] = orig + eps
        f_plus = float(f())
        flat[i] = orig - eps
        f_minus = float(f())
        flat[i] = orig
        gflat[i] = (f_plus - f_minus) / (2 * eps)
    return grad


def rel_error(a, b):
    return ((a - b).abs().max() / (a.abs() + b.abs()).max().clamp_min(1e-12)).item()


torch.manual_seed(0)
net = MLP([4, 6, 3])
net_linears = [layer for layer in net.layers if isinstance(layer, Linear)]
for layer in net_linears:
    layer.W, layer.b = layer.W.double(), (0.1 * torch.randn_like(layer.b)).double()
xs = torch.randn(5, 4, dtype=torch.float64)
ts = torch.randint(0, 3, (5,))


def loss_fn():
    return softmax_cross_entropy(net.forward(xs), ts)[0]


with torch.no_grad():
    _, g = softmax_cross_entropy(net.forward(xs), ts)
    net.backward(g)
    for k, layer in enumerate(net_linears):
        e_W = rel_error(layer.grad_W, numerical_grad(loss_fn, layer.W))
        e_b = rel_error(layer.grad_b, numerical_grad(loss_fn, layer.b))
        print(f"layer {k}: relative error grad_W {e_W:.1e}, grad_b {e_b:.1e}")
        assert e_W < 1e-6 and e_b < 1e-6

In [ ]:
# Training curve on the spirals (same setup as Test 5), printed every 50 steps
torch.manual_seed(1)
model = MLP([2, 32, 32, 2])
with torch.no_grad():
    for step in range(301):
        loss, grad = softmax_cross_entropy(model.forward(X), y)
        if step % 50 == 0:
            acc = (model.forward(X).argmax(dim=1) == y).float().mean().item()
            print(f"step {step:3d}   loss {float(loss):.4f}   accuracy {acc:.1%}")
        model.backward(grad)
        model.step(lr=0.5)

## Why it works

**Linear.** With $y = xW^\top + b$, i.e. $y_{no} = \sum_i x_{ni} W_{oi} + b_o$:

$$\frac{\partial L}{\partial W_{oi}} = \sum_n \frac{\partial L}{\partial y_{no}}\, x_{ni},\qquad
\frac{\partial L}{\partial b_o} = \sum_n \frac{\partial L}{\partial y_{no}},\qquad
\frac{\partial L}{\partial x_{ni}} = \sum_o \frac{\partial L}{\partial y_{no}}\, W_{oi},$$

i.e. `grad_W = grad_out.T @ x`, `grad_b = grad_out.sum(0)`, `grad_x = grad_out @ W`.

The bias is broadcast over the batch in the forward pass, so its gradient is a *sum* over the batch (broadcast forward ⇔ reduce backward).

**ReLU.** The derivative is $\mathbb{1}[x > 0]$. At $x = 0$ ReLU is not differentiable; any value in $[0, 1]$ is a valid subgradient and PyTorch picks 0.

**Why the softmax-CE gradient is just $p - y$.** For one example, $\ell = -\log p_y = -z_y + \log\sum_j e^{z_j}$, so

$$\frac{\partial \ell}{\partial z_k} = -\mathbb{1}[k = y] + \frac{e^{z_k}}{\sum_j e^{z_j}} = p_k - \mathbb{1}[k = y].$$

Going the long way round gives the same thing: the softmax Jacobian is $\partial p_i/\partial z_k = p_i(\delta_{ik} - p_k)$ and $\partial\ell/\partial p_i = -\delta_{iy}/p_i$, so
$\sum_i (-\delta_{iy}/p_i)\, p_i(\delta_{ik} - p_k) = p_k - \delta_{yk}$ — the $1/p_y$ from the log cancels the $p_y$ from the softmax Jacobian. That cancellation is why
frameworks fuse the two ops: the fused gradient never divides by a tiny probability, and it never saturates (compare sigmoid + MSE, whose gradient carries a
$p(1-p)$ factor that vanishes exactly when the model is confidently wrong). With mean reduction, each row is divided by $N$.

**Stability.** $\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}$ with $m = \max_j z_j$: every exponent is $\le 0$ (no overflow) and the largest term is
$e^0 = 1$, so the sum is $\ge 1$ (no $\log 0$). Computing `softmax` first and then `log` fails the other way: $p_y$ underflows to 0 and the loss becomes $+\infty$.

## Complexity

A Linear layer's forward is one matmul, $O(N \cdot d_\text{in} \cdot d_\text{out})$; its backward is two (`grad_W` and `grad_x`), so a training step costs ≈ 3× a
forward pass — the origin of the "6 × params FLOPs per token" rule for training. Memory: every layer's input (and ReLU's 1-bit mask) is cached until backward,
$O(N \sum_\ell d_\ell)$ — the activation memory that gradient checkpointing trades for recomputation.

## Common bugs

- `grad_W = x.T @ grad_out` — shape `(in, out)`, the transpose of the weight layout.
- Taking the *mean* of `grad_out` for `grad_b` (or dividing by $N$ again in the layers): the $1/N$ already lives in the loss gradient.
- `exp(z)` without subtracting the max → `inf / inf = NaN`; `log(softmax(z))` → `log(0) = -inf`.
- ReLU mask `x >= 0` (gradient 1 at the kink — differs from PyTorch).
- A ReLU after the last Linear: logits can no longer be negative, and dead units block the gradient.
- Accumulating `grad_W += ...` across steps without zeroing — PyTorch's `.grad` accumulates too, which is why `optimizer.zero_grad()` exists.
- Calling `backward` on the layers in forward order, or caching the input in `backward` instead of `forward`.

## Interview follow-ups

1. **Why is backward ~2× the FLOPs of forward?** Each Linear needs two matmuls in backward (w.r.t. weights and w.r.t. input) vs one in forward. So training ≈ 6·params FLOPs per token (2 forward + 4 backward), inference ≈ 2·params.
2. **What must be stored for backward, and how can you cut it?** Each Linear's input and each ReLU's mask (1 bit/element). Gradient checkpointing stores only some activations and recomputes the rest. ReLU's backward needs only its *output* ($y > 0 \Leftrightarrow x > 0$), which is why `nn.ReLU(inplace=True)` may overwrite its input — unless the previous op saved that tensor for its own backward (e.g. `sigmoid` and `tanh` save their output), in which case autograd raises "a variable needed for gradient computation has been modified by an inplace operation".
3. **How do you verify a hand-written gradient?** Central finite differences in float64 (relative error ≲ 1e-7), random coordinates for large tensors, away from kinks — or `torch.autograd.gradcheck`. In float32 round-off alone is ~1e-2, too coarse to catch subtle bugs.
4. **Why He init instead of Xavier?** Xavier ($\mathrm{Var} = 2/(d_\text{in}+d_\text{out})$) assumes a roughly linear, zero-symmetric activation; ReLU zeroes half its inputs and halves the second moment each layer, which He's factor 2 compensates. (PyTorch's `nn.Linear` default is `kaiming_uniform_(a=√5)` = $\mathcal U(\pm 1/\sqrt{d_\text{in}})$, i.e. $\mathrm{Var} = 1/(3 d_\text{in})$ — 6× smaller than He's $2/d_\text{in}$.)
5. **How does the gradient change with label smoothing or class weights?** With a smoothed target distribution $q$, the gradient is $(p - q)/N$. With class weights, row $n$ becomes $w_{y_n}(p_n - \text{onehot}_n)$ and PyTorch's mean divides by $\sum_n w_{y_n}$, not $N$.
6. **What changes for a sigmoid/BCE output or for MSE?** Sigmoid + BCE also collapses to $(p - y)/N$ w.r.t. the logit (same cancellation). MSE on logits gives $2(z - y)/N$; MSE on probabilities multiplies by the softmax/sigmoid Jacobian and saturates.